# 21_GameSkeleton_v2 — Khung xương game oẳn tù tì (bản sửa luật)

Model: `step2 s1` (`Data_321/checkpoints_task9/step2/seed1/TCN.pt`). Chỉ chạy model, **không train**. Bản cũ `20_GameSkeleton.ipynb` giữ nguyên.

**Cách chạy:** kích hoạt `C:\envs\train-gpu\Scripts\activate.bat`, mở notebook bằng kernel `train-gpu`, chạy lần lượt cell 1 → 10. Cell 6 tự kiểm tra luật (dữ liệu giả), cell 7 kiểm tra luật đọc hình tay trên dữ liệu thật, cell 10 mở webcam.

**Phím:** `SPACE` chơi / chơi tiếp | `r` reset điểm | `q` thoát.

**Luật mới:**
1. SPACE → đếm 3, 2, 1 (mỗi nhịp có tiếng tick). Mất tay quá `lost_max` giây thì dừng, nhắc, có tay lại thì đếm lại từ 3.
2. Tại "ra" phát tiếng khác và hiện chữ RA!; đúng `reveal_delay` = 0.1 giây sau đó AI chốt đáp án từ xác suất của model (trung bình `K` khung gần nhất) rồi ra quân khắc chế, và **khung hình lúc chốt là bằng chứng**, không chờ gì thêm. Cử chỉ người chơi được đọc **ngay trên khung bằng chứng** bằng luật ngón tay (cell 4), độc lập với model, nên thắng/thua là thật chứ không phải model tự chấm chính nó.
3. Kết quả hiện to, kèm khung bằng chứng đã đóng băng, và **giữ nguyên cho đến khi bấm SPACE** để chơi tiếp. Hình tay chỉ báo "không xác định" (không tính điểm) khi mất tay, hoặc khi bạn đặt `margin` > 0 trong `hand_shape`.

In [ ]:
#cell 1: import, đường dẫn, kiểm tra file cần có, bật CUDA
import time, math, collections
import numpy as np, torch, cv2
import torch.nn as nn, torch.nn.functional as F
import mediapipe as mp
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA; notebook này không chạy trên CPU."
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
TASK = BASE / "hand_landmarker.task"
CKPT = BASE / "checkpoints_task9" / "step2" / "seed1" / "TCN.pt"      # step2 s1
for p in (TASK, CKPT):
    assert p.exists(), f"Không thấy: {p}"
CLASSES = ["paper", "rock", "scissors"]                              # nhãn 0, 1, 2
print("OK | GPU:", torch.cuda.get_device_name(0))

In [ ]:
#cell 2: TCN + 546 đặc trưng (giống 13/19), nạp checkpoint step2 s1, làm nóng GPU
class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d
        self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)])
        self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)    # 210 cặp khớp

def features_from_pos(pos, hand_mask):
    """pos (b,T,21,3) -> (b,T,546): 126 gốc + 210 khoảng cách + 210 vận tốc khoảng cách (giống 13)."""
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    b, T = pos.shape[:2]
    flat = pos.reshape(b, T, 63)
    pv = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    vel = torch.zeros_like(flat); vel[:, 1:] = (flat[:, 1:] - flat[:, :-1]) * pv
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I] - pos[:, :, PAIR_J], dim=-1) * hand_mask[:, :, None]
    dvel = torch.zeros_like(dist); dvel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pv
    return torch.cat([flat, vel, dist, dvel], dim=-1)

def geo_from_base(X, hand_mask):
    return features_from_pos(X[..., :63].reshape(X.shape[0], X.shape[1], 21, 3), hand_mask)

net = TCN(in_features=546).to(dev)
ck = torch.load(CKPT, map_location=dev, weights_only=False)
assert ck.get("cfg", {}).get("n_in") == 546, f"Checkpoint không phải họ step2: {ck.get('cfg')}"
net.load_state_dict(ck["w"]); net.eval()
with torch.no_grad(): net(torch.zeros(1, 128, 546, device=dev))
print(f"Đã nạp step2 s1 | epoch {ck['epoch']} | val_loss {ck['val_loss']:.4f}")

In [ ]:
#cell 3: MediaPipe (CPU), nội suy webcam -> 120 fps, hàm trả xác suất 3 lớp của khung cuối (giống 13/19)
landmarker = mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(TASK)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO, num_hands=1))
T0 = time.perf_counter()          # timestamp MediaPipe phải luôn tăng, kể cả khi chạy lại cell 10

def detect(frame_bgr, ts_ms):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(mp.ImageFormat.SRGB, rgb), ts_ms)
    if not res.hand_landmarks:
        return None
    return np.array([[p.x, p.y, p.z] for p in res.hand_landmarks[0]], np.float32)

FPS_MODEL, WIN = 120, 128
buf = collections.deque(maxlen=WIN)
state = {"prev_lm": None, "prev_t": None}

def push(lm, t):
    prev, pt = state["prev_lm"], state["prev_t"]
    n = 1 if pt is None else max(1, round((t - pt) * FPS_MODEL))
    if lm is None:
        buf.extend([np.full((21, 3), np.nan, np.float32)] * n)
    elif prev is None:
        buf.append(lm)
    else:
        buf.extend(prev + (lm - prev) * (k / n) for k in range(1, n + 1))
    state["prev_lm"], state["prev_t"] = lm, t

def base_features(lm):
    has = ~np.isnan(lm[:, 0, 0])
    q = lm - lm[:, :1]
    q = np.nan_to_num(q / np.maximum(np.linalg.norm(q[:, 9], axis=1)[:, None, None], 1e-8)).astype(np.float32)
    q[~has] = 0
    pos = q.reshape(len(q), 63)
    vel = np.zeros_like(pos)
    if len(pos) > 1:
        vel[1:] = (pos[1:] - pos[:-1]) * (has[1:] & has[:-1])[:, None]
    return np.hstack([pos, vel]).astype(np.float32), has

def probs_now():
    """Xác suất (paper, rock, scissors) của khung cuối; None nếu khung cuối mất tay."""
    lm = np.stack(buf)
    if np.isnan(lm[-1, 0, 0]):
        return None
    base, has = base_features(lm)
    xb, mb = torch.from_numpy(base)[None].to(dev), torch.from_numpy(has)[None].to(dev)
    with torch.no_grad():
        return net(geo_from_base(xb, mb)).softmax(-1)[0, -1].cpu().numpy()
print("Sẵn sàng: MediaPipe + nội suy 120 fps + hàm probs_now()")

In [ ]:
#cell 4: luật đọc hình tay ngay trên 1 khung (độc lập với TCN), chấm điểm mềm nên khung nào có tay cũng ra đáp án
TIP, PIP = [8, 12, 16, 20], [6, 10, 14, 18]          # trỏ, giữa, áp út, út (bỏ ngón cái)
SIGN = np.array([[1, 1, 1, 1], [-1, -1, -1, -1], [1, 1, -1, -1]], np.float32)   # mẫu ngón duỗi(+)/gập(-): paper, rock, scissors

def finger_scores(lm):
    """lm (...,21,3) -> (...,4): log(đầu ngón / khớp giữa ngón) tính từ cổ tay; > 0 là duỗi, < 0 là gập, càng xa 0 càng chắc."""
    d = np.linalg.norm(lm - lm[..., :1, :], axis=-1)
    return np.log((d[..., TIP] + 1e-6) / (d[..., PIP] + 1e-6))

def hand_shape(lm, margin=0.0):
    """0 paper | 1 rock | 2 scissors; -1 chỉ khi mất tay (hoặc 2 lớp đứng đầu cách nhau < margin, mặc định 0 = luôn chọn một lớp)."""
    score = finger_scores(lm) @ SIGN.T                                    # (...,3) điểm khớp với từng mẫu
    top = np.sort(score, axis=-1)
    out = np.where(top[..., -1] - top[..., -2] < margin, -1, score.argmax(-1))
    return np.where(np.isnan(lm).any(axis=(-1, -2)), -1, out)
print("Đã định nghĩa hand_shape.")

In [ ]:
#cell 5: luật game (máy trạng thái, không dính tới webcam nên test được riêng ở cell 6)
COUNTER = {0: 2, 1: 0, 2: 1}      # người ra paper -> AI ra scissors; rock -> paper; scissors -> rock

class Game:
    def __init__(self, K=1, count_sec=1.0, lost_max=0.3, thr=0.6, reveal_delay=0.1):
        self.K, self.count_sec, self.lost_max, self.thr = K, count_sec, lost_max, thr
        self.reveal_delay = reveal_delay                          # đáp án ra sau nhịp "ra" bấy nhiêu giây
        self.ra_called = self.ra_now = False
        self.score = {"ai": 0, "player": 0, "draw": 0}
        self.state, self.msg, self.number, self.locked_now = "WAIT", "Nhan SPACE de choi", 3, False
        self.guess = self.ai = self.player = self.outcome = None

    def start(self, t, probs):
        """Bấm SPACE: chơi ván đầu hoặc chơi tiếp sau khi xem kết quả. Chưa có tay thì chờ tay rồi mới đếm."""
        if self.state not in ("WAIT", "RESULT"):
            return
        self.guess = self.ai = self.player = self.outcome = None
        if probs is None:
            self.state, self.msg = "NEED_HAND", "Dua tay vao khung hinh"
        else:
            self._begin_count(t, probs)

    def _begin_count(self, t, probs):
        self.state, self.msg, self.number, self.ra_called = "COUNT", "", 3, False
        self.t0, self.lost_since = t, None
        self.hist = collections.deque([probs], maxlen=self.K)     # K khung có tay gần nhất

    def step(self, t, probs, shape):
        """Gọi mỗi khung webcam. probs: 3 xác suất (None nếu mất tay); shape: hình tay đọc bằng luật (None nếu mất tay)."""
        self.locked_now = self.ra_now = False                     # locked_now: đúng khung chốt đáp án | ra_now: đúng khung nhịp "ra"
        if self.state == "NEED_HAND" and probs is not None:
            self._begin_count(t, probs)                           # có tay lại -> đếm lại từ 3
        elif self.state == "COUNT":
            self._count(t, probs, shape)

    def _count(self, t, probs, shape):
        if probs is None:
            self.lost_since = self.lost_since if self.lost_since is not None else t
            if t - self.lost_since > self.lost_max:               # mất tay quá lâu -> dừng đếm
                self.state, self.msg = "NEED_HAND", "Mat tay! Dua tay vao khung hinh"
                return
        else:
            self.lost_since = None
            self.hist.append(probs)
        e = t - self.t0
        if e >= 3 * self.count_sec and not self.ra_called:
            self.ra_called = self.ra_now = True                   # nhịp "ra" (tiếng "ra" phát ở đây)
        if e < 3 * self.count_sec + self.reveal_delay or probs is None:   # đáp án ra sau "ra" 0.1 s, chỉ ở khung CÓ tay (cần bằng chứng)
            self.number = max(1, 3 - int(e / self.count_sec))
            return
        p = np.mean(self.hist, axis=0)                            # chốt đáp án ngay, không random
        self.guess, self.ai = int(p.argmax()), COUNTER[int(p.argmax())]
        self.player = int(shape)                                  # bằng chứng: hình tay đọc trên chính khung chốt này
        d = None if self.player < 0 else (self.ai - self.player) % 3
        self.outcome = "unknown" if d is None else ("draw", "player", "ai")[d]
        if d is not None:
            self.score[self.outcome] += 1
        self.state, self.locked_now = "RESULT", True
print("Đã định nghĩa Game.")

In [ ]:
#cell 6: tự kiểm tra luật game và hand_shape bằng dữ liệu giả (không cần webcam)
rock, dt = np.array([.05, .90, .05]), 1 / 30

def run(g, probs, shape, n):
    for _ in range(n):
        run.t += dt; g.step(run.t, probs, shape)

def until(g, probs, shape, n=200):
    for _ in range(n):
        run.t += dt; g.step(run.t, probs, shape)
        if g.state == "RESULT": return
    raise AssertionError(f"Không tới được RESULT, đang ở {g.state}")

# 1) nhịp "ra" ở giây 3, đáp án chốt sau đó >= 0.1 s: AI đoán rock, ra paper; người ra rock -> AI thắng
run.t = 0.0; g = Game(); g.start(run.t, rock); t_ra = None
for _ in range(200):
    run.t += dt; g.step(run.t, rock, 1)
    if g.ra_now: assert t_ra is None; t_ra = run.t                  # nhịp "ra" chỉ xảy ra đúng 1 lần
    if g.state == "RESULT": break
assert 2.9 < t_ra < 3.1 and run.t > t_ra and run.t >= 3.1 - 1e-9, (t_ra, run.t)
assert g.guess == 1 and g.ai == 0 and g.locked_now and g.outcome == "ai"
run(g, rock, 1, 5); assert g.state == "RESULT" and not g.locked_now and g.score["ai"] == 1   # giữ kết quả, không tự thoát

# 2) người ra scissors ngay khung "ra" (AI đoán rock, ra paper) -> người thắng
run.t = 0.0; g = Game(); g.start(run.t, rock); until(g, rock, 2); assert g.outcome == "player"

# 3) không đọc được hình tay ở khung "ra" -> không xác định, không tính điểm
run.t = 0.0; g = Game(); g.start(run.t, rock); until(g, rock, -1)
assert g.outcome == "unknown" and sum(g.score.values()) == 0

# 4) SPACE ở màn kết quả -> chơi tiếp (đếm lại từ 3, điểm giữ nguyên)
g.start(run.t, rock); assert g.state == "COUNT" and g.number == 3 and g.outcome is None and not g.ra_called

# 5) mất tay ngắn (0.2 s) vẫn đếm; mất lâu thì dừng, có tay lại thì đếm lại từ 3
run.t = 0.0; g = Game(); g.start(run.t, rock)
run(g, None, None, 6); assert g.state == "COUNT"
run(g, None, None, 10); assert g.state == "NEED_HAND"
run(g, rock, 1, 1); assert g.state == "COUNT" and g.number == 3

# 6) mất tay đúng lúc "ra": chưa chốt (cần bằng chứng), có tay lại trong 0.3 s thì chốt
run.t = 0.0; g = Game(); g.start(run.t, rock); run(g, rock, 1, 88)
run(g, None, None, 3); assert g.state == "COUNT"
until(g, rock, 1); assert g.locked_now

# 7) bấm SPACE khi chưa có tay -> chờ tay
run.t = 0.0; g = Game(); g.start(run.t, None); assert g.state == "NEED_HAND"

# 8) hand_shape trên bàn tay giả: đầu ngón xa/gần cổ tay hơn khớp giữa ngón
def fake(ext):
    lm = np.zeros((21, 3), np.float32)
    for e, t, p in zip(ext, TIP, PIP): lm[p, 1], lm[t, 1] = 1.0, (1.5 if e else 0.7)
    return lm
assert hand_shape(fake([1, 1, 1, 1])) == 0 and hand_shape(fake([0, 0, 0, 0])) == 1
assert hand_shape(fake([1, 1, 0, 0])) == 2 and hand_shape(fake([1, 0, 0, 0]), margin=0.5) == -1
noisy = fake([1, 1, 0, 0]); noisy[16, 1] = 1.05                      # scissors nhưng ngón áp út duỗi nhẹ (luật cũ báo "?")
assert hand_shape(noisy) == 2
bad = fake([1, 1, 1, 1]); bad[:] = np.nan; assert hand_shape(bad) == -1
assert list(hand_shape(np.stack([fake([1, 1, 1, 1]), fake([0, 0, 0, 0])]))) == [0, 1]
print("Cả 8 phép thử đều đạt.")

In [ ]:
#cell 7: kiểm tra luật đọc hình tay trên dữ liệu thật (25% khung cuối mỗi video, lúc cử chỉ đã thành hình)
import pandas as pd
MAN = BASE / "manifest_all.csv"
assert MAN.exists(), f"Không thấy: {MAN}"
m = pd.read_csv(MAN)
cm = np.zeros((3, 4), int)                          # hàng: nhãn thật | cột: paper, rock, scissors, không rõ
for f, y in zip(m.path, m.nhan):
    assert (BASE / f).exists(), f"Thiếu file landmark: {BASE / f}"
    lm = np.load(BASE / f)["lm"]
    lm = lm[int(len(lm) * 0.75):]
    lm = lm[~np.isnan(lm[:, 0, 0])]                 # bỏ khung mất tay
    if len(lm):
        cm[int(y)] += np.bincount(np.where(hand_shape(lm) < 0, 3, hand_shape(lm)), minlength=4)
print(pd.DataFrame(cm, index=[f"thật: {c}" for c in CLASSES], columns=CLASSES + ["không rõ"]))
print(f"Đúng {np.trace(cm[:, :3]) / cm.sum():.1%} | không rõ {cm[:, 3].sum() / cm.sum():.1%} | sai {1 - (np.trace(cm[:, :3]) + cm[:, 3].sum()) / cm.sum():.1%} (trên {cm.sum()} khung)")

In [ ]:
#cell 8: giao diện (chữ đen trên nền màu, chữ không dấu vì OpenCV không vẽ được dấu); kết quả hiện to trên khung bằng chứng
COLORS = {0: (60, 60, 255), 1: (255, 150, 40), 2: (60, 210, 60)}      # BGR: paper đỏ, rock xanh dương, scissors xanh lá
GRAY, YELLOW, BLACK = (190, 190, 190), (150, 255, 255), (0, 0, 0)
BONES = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
         (9,13),(13,14),(14,15),(15,16),(13,17),(17,18),(18,19),(19,20),(0,17)]
BANNER = {"ai": ("AI THANG", (60, 60, 255)), "player": ("BAN THANG", (60, 210, 60)),
          "draw": ("HOA", YELLOW), "unknown": ("KHONG XAC DINH", GRAY)}

def put_label(img, text, x, y, bg, scale=0.8):
    (w, h), base = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.rectangle(img, (x, y - h - base - 6), (x + w + 10, y + 2), bg, -1)
    cv2.putText(img, text, (x + 5, y - base), cv2.FONT_HERSHEY_SIMPLEX, scale, BLACK, 2, cv2.LINE_AA)

def draw_hand(img, lm, color, mirror=True):
    h, w = img.shape[:2]
    px, py = (lm[:, 0] * w).astype(int), (lm[:, 1] * h).astype(int)
    if mirror: px = w - 1 - px
    for a, b in BONES:
        cv2.line(img, (px[a], py[a]), (px[b], py[b]), color, 2, cv2.LINE_AA)
    cv2.rectangle(img, (px.min() - 12, py.min() - 12), (px.max() + 12, py.max() + 12), color, 2)

def draw_game(img, g):
    h, w = img.shape[:2]
    if g.state == "COUNT":
        put_label(img, "RA!" if g.ra_called else str(g.number), w // 2 - (70 if g.ra_called else 30), h // 2 + 40, YELLOW, 4)
    elif g.state == "RESULT":
        text, col = BANNER[g.outcome]
        put_label(img, text, 10, 140, col, 1.8)
        if g.outcome == "unknown":
            put_label(img, "Khong doc duoc hinh tay luc AI chot", 10, h - 90, GRAY, 0.9)
        else:
            put_label(img, f"BAN ra: {CLASSES[g.player]}", 10, h - 130, COLORS[g.player], 1.0)
            put_label(img, f"AI doan ban ra {CLASSES[g.guess]} -> AI ra: {CLASSES[g.ai]}", 10, h - 90, COLORS[g.ai], 1.0)
        put_label(img, "Khung bang chung luc AI chot | SPACE: choi tiep", 10, h - 50, YELLOW, 0.7)
    else:
        put_label(img, g.msg, 10, h - 50, YELLOW, 0.9)
    s = g.score
    put_label(img, f"AI {s['ai']} - {s['player']} Nguoi | hoa {s['draw']}", 10, h - 10, YELLOW, 0.6)
print("Đã định nghĩa giao diện.")

In [ ]:
#cell 9: âm thanh: tick cho nhịp 3-2-1, "ra" cao và dài hơn; file wav tự tạo trong Data_321\sounds (có rồi thì không tạo lại)
import wave, winsound
SND = BASE / "sounds"; SND.mkdir(exist_ok=True)

def make_wav(path, freq, ms, vol=0.6):
    sr = 44100; t = np.arange(int(sr * ms / 1000)) / sr
    x = np.sin(2 * np.pi * freq * t) * np.minimum(1, np.minimum(t, t[-1] - t) * 200)   # mờ đầu/cuối, tránh tiếng "tạch"
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes((x * vol * 32767).astype(np.int16).tobytes())

SOUNDS = {"tick": (SND / "tick.wav", 800, 120), "ra": (SND / "ra.wav", 1400, 350)}
for path, f, ms in SOUNDS.values():
    if not path.exists(): make_wav(path, f, ms)

def play(name):                                       # phát không chặn vòng lặp webcam
    winsound.PlaySound(str(SOUNDS[name][0]), winsound.SND_FILENAME | winsound.SND_ASYNC)
print("Âm thanh ở:", SND)

In [ ]:
#cell 10: chạy webcam + game — SPACE: chơi / chơi tiếp | r: reset điểm | q: thoát
CAM_ID, MIRROR = 0, True
cap = cv2.VideoCapture(CAM_ID, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*"MJPG"))
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480); cap.set(cv2.CAP_PROP_FPS, 30)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
assert cap.isOpened(), "Không mở được webcam (thử đổi CAM_ID sang 1)."

buf.clear(); state.update(prev_lm=None, prev_t=None)
g = Game()                                    # đổi tham số luật ở đây, ví dụ Game(K=2, lost_max=0.5, reveal_delay=0.15)
evidence, fps, last = None, 0.0, time.perf_counter()
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        now = time.perf_counter()
        lm = detect(frame, int((now - T0) * 1000))
        push(lm, now)
        pr = probs_now() if lm is not None else None
        sh = int(hand_shape(lm)) if lm is not None else None
        before = (g.state, g.number)
        g.step(now, pr, sh)                   # luật chạy trước khi vẽ nên đáp án AI hiện ngay ở khung chốt
        if g.ra_now: play("ra")                # tiếng "ra" phát đúng nhịp; đáp án hiện sau reveal_delay (0.1 s)
        elif g.state == "COUNT" and (g.state, g.number) != before: play("tick")

        disp = cv2.flip(frame, 1) if MIRROR else frame
        if pr is not None:
            c, conf = int(pr.argmax()), float(pr.max())
            col = COLORS[c] if conf >= g.thr else GRAY
            draw_hand(disp, lm, col, MIRROR)
            put_label(disp, f"model: {CLASSES[c]}{'' if conf >= g.thr else '?'} {conf:.0%} | hinh: {CLASSES[sh] if sh >= 0 else '?'}", 10, 70, col, 0.6)
        if g.locked_now:
            evidence = disp.copy()            # bằng chứng: đúng khung chốt đáp án (có khung xương và nhãn)
        if g.state == "RESULT" and evidence is not None:
            disp = evidence.copy()            # màn kết quả đứng yên trên khung bằng chứng cho đến khi bấm SPACE
        fps, last = 0.9 * fps + 0.1 / max(now - last, 1e-6), now
        put_label(disp, f"FPS {fps:.0f}", 10, 30, YELLOW, 0.6)
        draw_game(disp, g)
        cv2.imshow("RPS game skeleton v2 (SPACE: choi | r: reset | q: thoat)", disp)

        k = cv2.waitKey(1) & 0xFF
        if k == ord("q"): break
        if k == 32:
            g.start(now, pr)
            if g.state == "COUNT": play("tick")
        if k == ord("r"): g.score = {"ai": 0, "player": 0, "draw": 0}
finally:
    cap.release(); cv2.destroyAllWindows()
print("Đã đóng webcam.")